### Instructions:

Using the Mental Health dataset and what you have learn this week, answer the following questions :

- What is the distribution of mental health conditions among different age groups in the tech industry?
- How does the frequency of mental health issues vary by gender?
- Identify the countries with the highest and lowest reported rates of mental health issues in the tech industry.

### Resources

Use the Mental Health in Tech Survey dataset available on Kaggle.

### Hint

1. Introduction to the Dataset:

    - Download the dataset from Kaggle.
    - Load the dataset using Pandas.
    - Perform initial exploration to understand the dataset structure : whats the distribution of the data? What types of data do i have?

2. Data Cleaning:

    - Identify and handle missing values.
    - Detect and correct any inconsistencies in the data.
    - Drop irrelevant columns if necessary.

In [229]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn preprocessing tools: StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.preprocessing import StandardScaler

# sklearn imputer: SimpleImputer
from sklearn.impute import SimpleImputer

df = pd.read_csv('data/survey.csv')

# ========== Basic Workflow ============

df.head()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


In [230]:
df.shape

(1259, 27)

In [231]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Timestamp                  1259 non-null   str  
 1   Age                        1259 non-null   int64
 2   Gender                     1259 non-null   str  
 3   Country                    1259 non-null   str  
 4   state                      744 non-null    str  
 5   self_employed              1241 non-null   str  
 6   family_history             1259 non-null   str  
 7   treatment                  1259 non-null   str  
 8   work_interfere             995 non-null    str  
 9   no_employees               1259 non-null   str  
 10  remote_work                1259 non-null   str  
 11  tech_company               1259 non-null   str  
 12  benefits                   1259 non-null   str  
 13  care_options               1259 non-null   str  
 14  wellness_program           1259 non

In [232]:
df.describe()

,Age
count,1.259000e+03
mean,7.942815e+07
std,2.818299e+09
min,-1.726000e+03
25%,2.700000e+01
50%,3.100000e+01
75%,3.600000e+01
max,1.000000e+11


**Observation**: Since the research questions focus on age, gender, country, and mental health treatment status, only the relevant columns were selected from the dataset. Irrelevant columns were excluded to simplify analysis and reduce noise.

In [233]:
# =========== DATA CLEANING ==============

# Check if there are any duplicates
print(f"Are there any duplicates in the orginal dataset? {df.duplicated().any()}")

relevant_cols = ["Age", "Gender", "Country", "treatment"]
df = df[relevant_cols]
print('\nReduced dataset (first five rows):')
print(df.head())

# Check if there are any nulls
print('\nAre there any null values in the reduced dataset?\n(True -> there are, False -> there aren\'t)')
df.isnull().any()

Are there any duplicates in the orginal dataset? False

Reduced dataset (first five rows):
   Age  Gender         Country treatment
0   37  Female   United States       Yes
1   44       M   United States        No
2   32    Male          Canada        No
3   31    Male  United Kingdom       Yes
4   31    Male   United States        No

Are there any null values in the reduced dataset?
(True -> there are, False -> there aren't)


Age          False
Gender       False
Country      False
treatment    False
dtype: bool

In [234]:
# Check for inconsistencies in the dataset

# Filter out garbage data (unrealistic/impossible ages)
before = len(df)

df = df[(df["Age"] >= 15) & (df["Age"] <= 80)]

after = len(df)

print("Removed rows:", before - after)



Removed rows: 8


In [235]:
# Check unique gender values
print("Values in 'Gender' column:")
print(df['Gender'].unique().tolist())

# Normalize strings
df["Gender"] = (
    df["Gender"]
      .str.strip()
      .str.lower()
)

# Group genders in ['male', 'female', 'other]
def clean_gender(g):
    if pd.isna(g):
        return "other"

    if "female" in g or g in ["f", "woman"]:
        return "female"

    if "male" in g or g in ["m", "man", "guy"]:
        return "male"

    return "other"


df["Gender"] = df["Gender"].apply(clean_gender)

Values in 'Gender' column:
['Female', 'M', 'Male', 'male', 'female', 'm', 'Male-ish', 'maile', 'Trans-female', 'Cis Female', 'F', 'something kinda male?', 'Cis Male', 'Woman', 'f', 'Mal', 'Male (CIS)', 'queer/she/they', 'non-binary', 'Femake', 'woman', 'Make', 'Nah', 'Enby', 'fluid', 'Genderqueer', 'Female ', 'Androgyne', 'Agender', 'cis-female/femme', 'Guy (-ish) ^_^', 'male leaning androgynous', 'Male ', 'Man', 'Trans woman', 'msle', 'Neuter', 'Female (trans)', 'queer', 'Female (cis)', 'Mail', 'cis male', 'Malr', 'femail', 'Cis Man', 'ostensibly male, unsure what that really means']


In [236]:
# Binary encoding of tratement
treatment_mapping = {'Yes': 1, 'No': 0}
df['treatment'] = df['treatment'].map(treatment_mapping)
df

,Age,Gender,Country,treatment
0,37,female,United States,1
1,44,male,United States,0
2,32,male,Canada,0
3,31,male,United Kingdom,1
4,31,male,United States,0
...,...,...,...,...
1254,26,male,United Kingdom,1
1255,32,male,United States,1
1256,34,male,United States,1
1257,46,female,United States,0


In [237]:
# Check unique countries values
print(df['Country'].unique().tolist())

['United States', 'Canada', 'United Kingdom', 'Bulgaria', 'France', 'Portugal', 'Netherlands', 'Switzerland', 'Poland', 'Australia', 'Germany', 'Russia', 'Mexico', 'Brazil', 'Slovenia', 'Costa Rica', 'Austria', 'Ireland', 'India', 'South Africa', 'Italy', 'Sweden', 'Colombia', 'Latvia', 'Romania', 'Belgium', 'New Zealand', 'Spain', 'Finland', 'Uruguay', 'Israel', 'Bosnia and Herzegovina', 'Hungary', 'Singapore', 'Japan', 'Nigeria', 'Croatia', 'Norway', 'Thailand', 'Denmark', 'Greece', 'Moldova', 'Georgia', 'China', 'Czech Republic', 'Philippines']


In [238]:
# Check unique treatment values
print(df['treatment'].unique().tolist())

[1, 0]


In [239]:
# Create age group feature
bins = [15, 25, 35, 45, 55, 65, 80]
labels = ["15–24", "25–34", "35–44", "45–54", "55–64", "65+"]

df["age_group"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels
)

# Create summary of distribution of mental health issues among age groups
age_summary = df.groupby("age_group")["treatment"].agg(
    total="size",
    with_issue="sum",
    percent="mean"
)
age_summary["percent"] = age_summary["percent"].round(2) * 100

print(age_summary)

           total  with_issue  percent
age_group                            
15–24        217         105     48.0
25–34        701         341     49.0
35–44        277         153     55.0
45–54         42          24     57.0
55–64         13           8     62.0
65+            1           1    100.0


In [240]:
# frequency of mental health issues by gender
gender_summary = df.groupby("Gender")["treatment"].agg(
    total="size",
    with_issue="sum",
    percent="mean"
)
gender_summary["percent"] = gender_summary["percent"].round(2) * 100

print(gender_summary)

        total  with_issue  percent
Gender                            
female    248         172     69.0
male      979         444     45.0
other      24          16     67.0


In [241]:
country_summary = df.groupby("Country")["treatment"].agg(
    total="size",
    with_issue="sum",
    percent="mean"
)
country_summary["percent"] = country_summary["percent"].round(2) * 100

country_sorted = country_summary.sort_values(by='with_issue', ascending=False)

print("\nCountries with the highest count of reported health issues:\n")
print(country_sorted.head(5))

print("\nCountries with the lowest count of reported health issues:\n")
print(country_sorted.tail(5))


Countries with the highest count of reported health issues:

                total  with_issue  percent
Country                                   
United States     746         408     55.0
United Kingdom    184          92     50.0
Canada             72          37     51.0
Germany            45          21     47.0
Australia          21          13     62.0

Countries with the lowest count of reported health issues:

         total  with_issue  percent
Country                            
Nigeria      1           0      0.0
Greece       2           0      0.0
Hungary      1           0      0.0
Israel       5           0      0.0
Uruguay      1           0      0.0
